In [25]:
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath("../../"))
from lib import io as lio

In [26]:
year = 2018

In [27]:
cbp = pd.read_csv(f"raw/cbp_{year}.csv")
cbp["county"] = cbp["fipstate"].astype(str).str.zfill(2) + cbp["fipscty"].astype(
    str
).str.zfill(3)
cbp

,fipstate,fipscty,naics,emp_nf,emp,qp1_nf,qp1,ap_nf,ap,est,...,n250_499,n500_999,n1000,n1000_1,n1000_2,n1000_3,n1000_4,censtate,cencty,county
0,1,1,------,G,11397,G,90886,G,373865,855,...,N,N,N,N,N,N,N,63,1,01001
1,1,1,11----,H,93,H,1176,H,4965,10,...,N,N,N,N,N,N,N,63,1,01001
2,1,1,113///,G,79,G,1017,H,4179,7,...,N,N,N,N,N,N,N,63,1,01001
3,1,1,1133//,G,79,G,1017,H,4179,7,...,N,N,N,N,N,N,N,63,1,01001
4,1,1,11331/,G,79,G,1017,H,4179,7,...,N,N,N,N,N,N,N,63,1,01001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1086175,56,999,61131/,G,17,H,50,G,190,3,...,N,N,N,N,N,N,N,83,999,56999
1086176,56,999,611310,G,17,H,50,G,190,3,...,N,N,N,N,N,N,N,83,999,56999
1086177,56,999,62----,H,25,G,232,G,761,3,...,N,N,N,N,N,N,N,83,999,56999
1086178,56,999,621///,H,25,G,232,G,761,3,...,N,N,N,N,N,N,N,83,999,56999


In [ ]:
# cbp data structure is nested; this encompasses all of sectors 71/72/81 (ent/fod/srv), the more specific columns partition this
# use 722/// since 711/// is accomodations (e.g., hotels, casinos, resorts)
naics_code_to_sector = {"71----": "ENT", "722///": "FOD", "81----": "SRV"}
amenities = cbp[cbp.naics.isin(naics_code_to_sector.keys())].copy()
amenities["naics_sector"] = amenities.naics.map(naics_code_to_sector)

In [29]:
# get for each county the amount of establishments in ent/fod/srv
wide = (
    amenities.pivot(index="county", columns="naics_sector", values="est")
    .fillna(0)
    .reset_index()
)
wide

naics_sector,county,ENT,FOD,SRV
0,01001,12.0,91.0,112.0
1,01003,96.0,500.0,501.0
2,01005,6.0,40.0,39.0
3,01007,5.0,17.0,35.0
4,01009,9.0,47.0,101.0
...,...,...,...,...
3157,56039,101.0,99.0,148.0
3158,56041,5.0,38.0,29.0
3159,56043,6.0,19.0,32.0
3160,56045,0.0,13.0,18.0


In [30]:
# read in the county to 2010 puma crosswalk
xwalk = pd.read_csv(
    "../geometry/equivalencies/county_to_2010_puma.csv",
    encoding="latin-1",
    skiprows=[1],
)
xwalk["county"] = xwalk["county"].astype(str).str.zfill(5)
xwalk["PUMA"] = xwalk["state"].astype(str).str.zfill(2) + xwalk["puma12"].astype(
    str
).str.zfill(5)
xwalk

,county,state,puma12,stab,cntyname,PUMAname,pop10,afact,PUMA
0,01001,1,2100,AL,Autauga AL,"Elmore, Autauga, Montgomery (Outer) & Lowndes ...",54571,1.0,0102100
1,01003,1,2600,AL,Baldwin AL,Baldwin County,182265,1.0,0102600
2,01005,1,2400,AL,Barbour AL,"Russell, Pike, Barbour, Macon & Bullock Counties",27457,1.0,0102400
3,01007,1,1700,AL,Bibb AL,"Dallas, Bibb, Marengo, Hale, Sumter, Perry & G...",22915,1.0,0101700
4,01009,1,800,AL,Blount AL,St. Clair & Blount Counties,57322,1.0,0100800
...,...,...,...,...,...,...,...,...,...
4541,56037,56,500,WY,Sweetwater WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",43806,1.0,5600500
4542,56039,56,100,WY,Teton WY,"Sheridan, Park, Teton, Lincoln & Big Horn Coun...",21294,1.0,5600100
4543,56041,56,500,WY,Uinta WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",21118,1.0,5600500
4544,56043,56,200,WY,Washakie WY,"Campbell, Goshen, Platte, Johnson, Washakie, W...",8533,1.0,5600200


In [31]:
# inject establishment count to the crosswalk
m = xwalk.merge(wide, on="county", how="left")
m

,county,state,puma12,stab,cntyname,PUMAname,pop10,afact,PUMA,ENT,FOD,SRV
0,01001,1,2100,AL,Autauga AL,"Elmore, Autauga, Montgomery (Outer) & Lowndes ...",54571,1.0,0102100,12.0,91.0,112.0
1,01003,1,2600,AL,Baldwin AL,Baldwin County,182265,1.0,0102600,96.0,500.0,501.0
2,01005,1,2400,AL,Barbour AL,"Russell, Pike, Barbour, Macon & Bullock Counties",27457,1.0,0102400,6.0,40.0,39.0
3,01007,1,1700,AL,Bibb AL,"Dallas, Bibb, Marengo, Hale, Sumter, Perry & G...",22915,1.0,0101700,5.0,17.0,35.0
4,01009,1,800,AL,Blount AL,St. Clair & Blount Counties,57322,1.0,0100800,9.0,47.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4541,56037,56,500,WY,Sweetwater WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",43806,1.0,5600500,12.0,93.0,109.0
4542,56039,56,100,WY,Teton WY,"Sheridan, Park, Teton, Lincoln & Big Horn Coun...",21294,1.0,5600100,101.0,99.0,148.0
4543,56041,56,500,WY,Uinta WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",21118,1.0,5600500,5.0,38.0,29.0
4544,56043,56,200,WY,Washakie WY,"Campbell, Goshen, Platte, Johnson, Washakie, W...",8533,1.0,5600200,6.0,19.0,32.0


In [32]:
cols = list(naics_code_to_sector.values())
# use the afact adjustment factor to scale the establish count, also population
m[cols] = m[cols].fillna(0).mul(m.afact, axis=0)
m

,county,state,puma12,stab,cntyname,PUMAname,pop10,afact,PUMA,ENT,FOD,SRV
0,01001,1,2100,AL,Autauga AL,"Elmore, Autauga, Montgomery (Outer) & Lowndes ...",54571,1.0,0102100,12.0,91.0,112.0
1,01003,1,2600,AL,Baldwin AL,Baldwin County,182265,1.0,0102600,96.0,500.0,501.0
2,01005,1,2400,AL,Barbour AL,"Russell, Pike, Barbour, Macon & Bullock Counties",27457,1.0,0102400,6.0,40.0,39.0
3,01007,1,1700,AL,Bibb AL,"Dallas, Bibb, Marengo, Hale, Sumter, Perry & G...",22915,1.0,0101700,5.0,17.0,35.0
4,01009,1,800,AL,Blount AL,St. Clair & Blount Counties,57322,1.0,0100800,9.0,47.0,101.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4541,56037,56,500,WY,Sweetwater WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",43806,1.0,5600500,12.0,93.0,109.0
4542,56039,56,100,WY,Teton WY,"Sheridan, Park, Teton, Lincoln & Big Horn Coun...",21294,1.0,5600100,101.0,99.0,148.0
4543,56041,56,500,WY,Uinta WY,"Sweetwater, Fremont, Uinta, Sublette & Hot Spr...",21118,1.0,5600500,5.0,38.0,29.0
4544,56043,56,200,WY,Washakie WY,"Campbell, Goshen, Platte, Johnson, Washakie, W...",8533,1.0,5600200,6.0,19.0,32.0


In [33]:
# aggregate establishments to the puma level
puma_est = m.groupby("PUMA")[cols].sum()
puma_est

,ENT,FOD,SRV
PUMA,,,
0100100,46.7864,310.0152,452.8016
0100200,43.8444,290.6388,359.8702
0100301,41.2032,260.0064,276.3456
0100302,35.1712,221.9424,235.8896
0100400,17.0000,145.0000,204.0000
...,...,...,...
5600100,207.0000,318.0000,444.0000
5600200,53.0000,226.0000,346.0000
5600300,56.0000,252.0000,397.0000


In [34]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma = puma_migpuma[~puma_migpuma["State"].astype(int).isin([2, 15, 72])]
puma_migpuma

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400
...,...,...
5600100,56,5600100
5600200,56,5600200
5600300,56,5600300


In [35]:
puma_migpuma[cols] = puma_est.loc[puma_migpuma.index, cols]

In [36]:
migpuma_est = puma_migpuma.groupby("MIGPUMA")[cols].sum()
migpuma_est

,ENT,FOD,SRV
MIGPUMA,,,
0100190,66.0000,471.0000,720.0000
0100290,154.9884,1003.9268,1105.9222
0100400,17.0000,145.0000,204.0000
0100600,28.0000,211.0000,337.0000
0100700,25.0000,163.0000,245.0000
...,...,...,...
5600100,207.0000,318.0000,444.0000
5600200,53.0000,226.0000,346.0000
5600300,56.0000,252.0000,397.0000


In [37]:
puma_est.to_csv(f"puma_est_{year}.csv")
migpuma_est.to_csv(f"migpuma_est_{year}.csv")